# ATLAS H→γγ Analysis with TaskVine (self-contained)

Everything needed to run the ATLAS Open Data H→γγ analysis with TaskVine, from scratch, in one notebook: a dedicated pixi kernel, downloading the dataset, submitting per-file TaskVine tasks through `taskvine-gateway`, and the final fit/plot - adapted from [the standalone TaskVine example](https://github.com/JinZhou5042/notebooks-collection-opendata/tree/hyy-taskvine-demo/13-TeV-examples/uproot_python/taskvine_executor/standalone) (itself adapted from [`HyyAnalysis.ipynb`](https://github.com/atlas-outreach-data-tools/notebooks-collection-opendata/blob/master/13-TeV-examples/uproot_python/HyyAnalysis.ipynb)).

**Two changes from the original demo**, both because self-service worker pools (`taskvine-gateway`) run on separate pods from the notebook, unlike the static pool the original demo assumed:

- **Data goes to the shared `/data/<shared-data-directory>` mount, not the notebook's own root disk.** Worker pods can't see the notebook pod's `/home/jovyan` at all - only the per-user `shared-data` PVC (mounted at `/data/<shared-data-directory>` on both sides) is common ground.
- **Each task ships its own environment via [poncho](https://cctools.readthedocs.io/en/latest/poncho/).** `taskvine-gateway`'s worker image only has `ndcctools` baked in - not `uproot`/`awkward`/`vector`/`numpy` - so every submitted task carries a packed environment with exactly those alongside it, rather than needing them preinstalled on the worker.

Run each cell to reproduce the analysis and plot. See this directory’s README for the result of a validated ODF run.

## Part 0: Build a kernel with everything this notebook needs

`ndcctools`, `taskvine-gateway`, and the ATLAS analysis stack together - see `custom-pixi-kernel.ipynb` for more on how this mechanism works. `pixi` itself isn't installed in this notebook's environment, so the first cell installs it.

In [ ]:
!curl -fsSL https://pixi.sh/install.sh | sh

In [ ]:
import os

# Relative to wherever this notebook itself is running from, not
# absolute under $HOME - keeps this kernel's files next to the notebook
# instead of scattered at the home directory root.
# %%writefile below needs this directory to already exist - it does a
# plain open(), it won't create missing parent directories itself.
os.makedirs("taskvine-atlas-kernel", exist_ok=True)

In [ ]:
%%writefile taskvine-atlas-kernel/pixi.toml
[workspace]
name = "taskvine-atlas-kernel"
channels = ["conda-forge"]
platforms = ["linux-64"]

[dependencies]
python = "3.11.*"
ndcctools = "==7.17.1"
ipykernel = "*"
aiohttp = "==3.14.3"
atlasopenmagic = "==1.9.1"
awkward = "==2.11.0"
fsspec = "==2026.6.0"
lmfit = "==1.3.2"
matplotlib = "==3.10.0"
numpy = "==2.4.6"
uproot = "==5.7.5"
vector = "==1.8.1"

[pypi-dependencies]
taskvine-gateway = { git = "https://github.com/maniaclab/taskvine-gateway", rev = "7be0706492e50ba262f86172eff66388f6fb8a4f" }


In [ ]:
import os

# Reference pixi by its absolute install path throughout this notebook
# rather than relying on PATH - simpler and more robust than fixing up
# PATH per kernel/session, and it works identically before and after
# switching kernels below.
PIXI = os.path.expanduser("~/.pixi/bin/pixi")
!{PIXI} install --manifest-path taskvine-atlas-kernel/pixi.toml

In [ ]:
!{PIXI} run --manifest-path taskvine-atlas-kernel/pixi.toml \
    python -m ipykernel install --user \
    --name taskvine-atlas --display-name "Python (taskvine-atlas)"

Reload this notebook's page (or **Kernel → Change Kernel**) and pick **Python (taskvine-atlas)** - a newly registered kernel doesn't always show up without a page refresh.

**Run everything below this point on that kernel.**

## Part 1: Write the analysis code

`hyy_analysis.py` (ROOT reading, event selections, histogram, cutflow, and result merging) is unchanged from the original demo - it's written to a real file both so this notebook can import it and so it can be shipped to each worker as a task input, exactly as the original does.

In [ ]:
%%writefile hyy_analysis.py
import numpy as np
import awkward as ak
import uproot
import vector


TREE_NAME = "analysis"
VARIABLES = [
    "photon_pt",
    "photon_eta",
    "photon_phi",
    "photon_e",
    "photon_isTightID",
    "photon_ptcone20",
]
BIN_EDGES = np.arange(100, 161, 1)


def cut_photon_reconstruction(is_tight):
    return is_tight[:, 0] & is_tight[:, 1]


def cut_photon_pt(pt):
    return (pt[:, 0] > 50) & (pt[:, 1] > 30)


def cut_isolation_pt(ptcone20, pt):
    return ((ptcone20[:, 0] / pt[:, 0]) < 0.055) & (
        (ptcone20[:, 1] / pt[:, 1]) < 0.055
    )


def cut_photon_eta_transition(eta):
    condition_0 = (np.abs(eta[:, 0]) < 1.52) | (np.abs(eta[:, 0]) > 1.37)
    condition_1 = (np.abs(eta[:, 1]) < 1.52) | (np.abs(eta[:, 1]) > 1.37)
    return condition_0 & condition_1


def calc_mass(pt, eta, phi, energy):
    p4 = vector.zip({"pt": pt, "eta": eta, "phi": phi, "e": energy})
    return (p4[:, 0] + p4[:, 1]).M


def cut_mass(mass):
    return mass != 0


def cut_iso_mass(pt, mass):
    return ((pt[:, 0] / mass) > 0.35) & ((pt[:, 1] / mass) > 0.35)


def process_file(path):
    tree = uproot.open(f"{path}:{TREE_NAME}")
    cutflow = np.zeros(7, dtype=np.int64)
    masses = []

    for data in tree.iterate(VARIABLES, library="ak"):
        cutflow[0] += len(data)
        data = data[cut_photon_reconstruction(data["photon_isTightID"])]
        cutflow[1] += len(data)
        data = data[cut_photon_pt(data["photon_pt"])]
        cutflow[2] += len(data)
        data = data[cut_isolation_pt(data["photon_ptcone20"], data["photon_pt"])]
        cutflow[3] += len(data)
        data = data[cut_photon_eta_transition(data["photon_eta"])]
        cutflow[4] += len(data)
        data["mass"] = calc_mass(
            data["photon_pt"],
            data["photon_eta"],
            data["photon_phi"],
            data["photon_e"],
        )
        data = data[cut_mass(data["mass"])]
        cutflow[5] += len(data)
        data = data[cut_iso_mass(data["photon_pt"], data["mass"])]
        cutflow[6] += len(data)
        masses.append(data["mass"])

    masses = ak.concatenate(masses) if masses else ak.Array([])
    histogram, _ = np.histogram(ak.to_numpy(masses), bins=BIN_EDGES)
    return {
        "histogram": histogram.astype(np.int64),
        "cutflow": cutflow,
        "entries": int(cutflow[0]),
        "selected": int(cutflow[-1]),
    }


def merge_results(results):
    return {
        "histogram": np.sum(
            [result["histogram"] for result in results], axis=0, dtype=np.int64
        ),
        "cutflow": np.sum(
            [result["cutflow"] for result in results], axis=0, dtype=np.int64
        ),
        "entries": sum(result["entries"] for result in results),
        "selected": sum(result["selected"] for result in results),
    }


## Part 2: Download the dataset to the shared `/data` mount

`download_data.py` is also unchanged - `download(data_dir)` takes the target directory as a parameter, so the only actual change is *which* directory this notebook passes it.

In [ ]:
%%writefile download_data.py
#!/usr/bin/env python3
"""Download the H->gammagamma inputs using the repository's cache mechanism."""

import argparse
from pathlib import Path
from urllib.parse import urlparse

import aiohttp
import atlasopenmagic as atom
import fsspec


def download(data_dir):
    """Download the GamGam skim into data_dir, returning the sorted local paths.

    Importable from a notebook, e.g. to target a scratch directory with more
    space than the JupyterHub home volume.
    """
    data_dir = Path(data_dir).resolve()
    data_dir.mkdir(parents=True, exist_ok=True)

    # Same atlasopenmagic + simplecache approach used by ../HyyAnalysis.ipynb.
    atom.set_release("2025e-13tev-beta")
    urls = atom.get_urls("data", "GamGam", protocol="https", cache=True)
    timeout = aiohttp.ClientTimeout(total=None, sock_connect=120, sock_read=3600)
    http = fsspec.filesystem("https", client_kwargs={"timeout": timeout})
    for index, url in enumerate(urls, 1):
        source = url.removeprefix("simplecache::")
        cached = data_dir / Path(urlparse(source).path).name
        if cached.exists() and cached.stat().st_size != http.info(source)["size"]:
            cached.unlink()
        print(f"downloading {index}/{len(urls)}", flush=True)
        fsspec.open_local(
            url,
            simplecache={
                "cache_storage": str(data_dir),
                "same_names": True,
            },
            https={"client_kwargs": {"timeout": timeout}},
        )
    files = sorted(data_dir.glob("*.root"))
    print(f"downloaded {len(files)} files to {data_dir}")
    return files


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-dir", type=Path, default=Path("data"))
    args = parser.parse_args()
    download(args.data_dir)


if __name__ == "__main__":
    main()


`/data` holds your own per-user shared-data directory - named after a k8s-safe slug of your JupyterHub username that KubeSpawner computes (not the username itself), possibly alongside other shared spaces too. There's no reliable way for this notebook to know which is yours - check the listing below and fill in your own directory name.

In [ ]:
!ls /data

In [ ]:
from pathlib import Path

from download_data import download

SHARED_DATA_DIR = "REPLACE_ME"  # fill in your own directory name from the `ls /data` output above

if SHARED_DATA_DIR == "REPLACE_ME":
    raise ValueError("Run `!ls /data` above and replace REPLACE_ME with your shared-data directory name")

DATA_DIR = Path(f"/data/{SHARED_DATA_DIR}/hyy-taskvine-data")
files = download(DATA_DIR)
print(f"{len(files)} ROOT files ready in {DATA_DIR}")

## Part 3: Package a worker environment with poncho

A separate, smaller pixi environment with just what `hyy_analysis.py` needs at import time (`uproot`, `awkward`, `vector`, `numpy`) plus `cloudpickle` (what `PythonTask` itself uses to serialize the submitted function - `ndcctools` pulls this in on the manager side, so the packed environment needs a matching copy for the worker side to unpickle it). `poncho_package_create` - which ships with `ndcctools`, so it runs via `pixi run --manifest-path` against *this* kernel's own environment, not the task environment's - packs an existing environment directory into a tarball workers unpack and activate on demand.

In [ ]:
import os

os.makedirs("hyy-task-env", exist_ok=True)

In [ ]:
%%writefile hyy-task-env/pixi.toml
[workspace]
name = "hyy-task-env"
channels = ["conda-forge"]
platforms = ["linux-64"]

[dependencies]
python = "3.11.*"
awkward = "==2.11.0"
numpy = "==2.4.6"
uproot = "==5.7.5"
vector = "==1.8.1"
cloudpickle = "==3.1.2"


In [ ]:
import os

# Redefine PIXI - this cell runs on the kernel this notebook switched
# to (Part 0, above), a fresh kernel session with no memory of the
# bootstrap kernel's own PIXI variable from before the switch.
PIXI = os.path.expanduser("~/.pixi/bin/pixi")
!{PIXI} install --manifest-path hyy-task-env/pixi.toml

# poncho_package_create ships with ndcctools, so it must run inside
# *this* kernel's own environment (via `pixi run --manifest-path`), not
# invoked directly by its own script path - its shebang is a plain
# `#!/usr/bin/env python3`, which re-resolves "python3" against ambient
# shell PATH rather than this environment specifically, and would find
# some other Python with no ndcctools installed at all.
TASK_ENV_TARBALL = "hyy-task-env.tar.gz"
!{PIXI} run --manifest-path taskvine-atlas-kernel/pixi.toml \
    poncho_package_create hyy-task-env/.pixi/envs/default {TASK_ENV_TARBALL}
print("packed task environment to", TASK_ENV_TARBALL)

## Part 4: Start the manager and request workers

In [ ]:
import ndcctools.taskvine as vine

m = vine.Manager(port=9123)
print(f"Manager listening on {m.port}")

In [ ]:
from taskvine_gateway import TaskVineCluster

cluster = TaskVineCluster()
cluster.scale(4, cores=1, memory_mb=2000)  # a small pool - TaskVine queues the 16 file-tasks across it
cluster.status()

In [ ]:
for _ in range(60):
    m.wait(1)   # actually services the manager's network I/O
    connected = m.stats.workers_connected
    print(f"workers connected: {connected}")
    if connected >= 4:
        break

## Part 5: Submit one task per file and collect results

Each task ships `hyy_analysis.py` as an input file (so `import hyy_analysis` resolves on the worker) and the packed poncho environment (so `uproot`/`awkward`/`vector`/`numpy` are actually importable there) - otherwise identical to the original demo's `hyy_taskvine.py`.

In [ ]:
def process_file_task(url):
    try:
        return {"status": "success", "result": process_file(url)}
    except BaseException as exc:
        return {
            "status": "failed",
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        }


def submit_file_tasks(manager, files, analysis_file, poncho_env):
    submitted = []
    for file_index, url in enumerate(files):
        task = vine.PythonTask(process_file_task, str(url))
        task.add_input(analysis_file, "hyy_analysis.py")
        task.add_environment(poncho_env)
        task.set_cores(1)
        task.set_tag(f"hyy-file-{file_index:02d}")
        manager.submit(task)
        submitted.append((task, file_index, url))
    return submitted


def collect_task_results(manager, submitted, timeout=60):
    results = {}
    while not manager.empty():
        task = manager.wait(timeout)
        if task is not None:
            results[task.tag] = task.output
    return results


def find_failed_tasks(submitted, results):
    return [
        {
            "tag": task.tag,
            "file_index": file_index,
            "url": str(url),
            "result": results.get(task.tag),
        }
        for task, file_index, url in submitted
        if not isinstance(results.get(task.tag), dict)
        or results[task.tag].get("status") != "success"
    ]

In [ ]:
import json
import time

from hyy_analysis import BIN_EDGES, merge_results, process_file

analysis_file = m.declare_file("hyy_analysis.py", cache=True)
poncho_env = m.declare_poncho(TASK_ENV_TARBALL, cache=True)

start_time = time.time()
submitted = submit_file_tasks(m, files, analysis_file, poncho_env)
results = collect_task_results(m, submitted)
failed = find_failed_tasks(submitted, results)
elapsed = time.time() - start_time

if failed:
    raise RuntimeError(f"{len(failed)} of {len(submitted)} tasks failed: {json.dumps(failed, default=str)}")

merged = merge_results([results[task.tag]["result"] for task, *_ in submitted])
merged["elapsed_seconds"] = elapsed
merged["tasks"] = len(submitted)
print(
    f"processed {merged['tasks']} tasks, "
    f"{merged['entries']} entries, {merged['selected']} selected, "
    f"in {merged['elapsed_seconds']:.1f}s"
)

In [ ]:
print("cutflow:", merged["cutflow"])

## Part 6: Fit and plot the invariant mass spectrum

Same 4th-order-polynomial-plus-Gaussian fit as the original demo, applied to the histogram merged from the TaskVine workers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, AutoMinorLocator
from lmfit.models import PolynomialModel, GaussianModel

xmin, xmax = 100, 160  # GeV
step_size = 1  # GeV

bin_edges = BIN_EDGES
bin_centres = (bin_edges[:-1] + bin_edges[1:]) / 2

data_x = merged["histogram"]
data_x_errors = np.sqrt(data_x)

# data fit
polynomial_mod = PolynomialModel(4)  # 4th order polynomial (background)
gaussian_mod = GaussianModel()  # Gaussian (signal)

pars = polynomial_mod.guess(
    data_x, x=bin_centres, c0=data_x.max(), c1=0, c2=0, c3=0, c4=0
)
pars += gaussian_mod.guess(data_x, x=bin_centres, amplitude=100, center=125, sigma=2)

model = polynomial_mod + gaussian_mod
out = model.fit(data_x, pars, x=bin_centres, weights=1 / data_x_errors)

params_dict = out.params.valuesdict()
c0, c1, c2, c3, c4 = (params_dict[f"c{i}"] for i in range(5))
background = c0 + c1 * bin_centres + c2 * bin_centres**2 + c3 * bin_centres**3 + c4 * bin_centres**4
signal_x = data_x - background

In [ ]:
# *************
# Main plot
# *************
plt.figure(figsize=(8, 8))
plt.axes([0.1, 0.3, 0.85, 0.65])  # left, bottom, width, height
main_axes = plt.gca()

main_axes.errorbar(
    x=bin_centres, y=data_x, yerr=data_x_errors,
    fmt="ko", label="Data", markersize=4,
)
main_axes.plot(bin_centres, out.best_fit, "-r", label="Sig+Bkg Fit ($m_H=125$ GeV)")
main_axes.plot(bin_centres, background, "--r", label="Bkg (4th order polynomial)")

main_axes.set_xlim(left=xmin, right=xmax)
main_axes.xaxis.set_minor_locator(AutoMinorLocator())
main_axes.tick_params(which="both", direction="in", top=True, labelbottom=False, right=True)
main_axes.set_ylabel("Events / " + str(step_size) + " GeV", horizontalalignment="right")
main_axes.set_ylim(bottom=0, top=np.amax(data_x) * 1.5)
main_axes.yaxis.set_minor_locator(AutoMinorLocator())
main_axes.yaxis.get_major_ticks()[0].set_visible(False)

plt.text(0.2, 0.92, "ATLAS Open Data", transform=main_axes.transAxes, fontsize=13)
plt.text(0.2, 0.86, "for education", transform=main_axes.transAxes, style="italic", fontsize=8)

lumi = 36.1
plt.text(
    0.2, 0.8,
    r"$\sqrt{s}$=13 TeV,$\int$L dt = " + str(lumi) + r" fb$^{-1}$",
    transform=main_axes.transAxes,
)
plt.text(0.2, 0.74, r"$H \rightarrow \gamma\gamma$", transform=main_axes.transAxes)

main_axes.legend(frameon=False, loc="lower left")

# *************
# Data-Bkg plot
# *************
plt.axes([0.1, 0.1, 0.85, 0.2])
sub_axes = plt.gca()

sub_axes.yaxis.set_major_locator(MaxNLocator(nbins="auto", symmetric=True))
sub_axes.errorbar(x=bin_centres, y=signal_x, yerr=data_x_errors, fmt="ko", markersize=4)
sub_axes.plot(bin_centres, out.best_fit - background, "-r")
sub_axes.plot(bin_centres, background - background, "--r")

sub_axes.set_xlim(left=xmin, right=xmax)
sub_axes.xaxis.set_minor_locator(AutoMinorLocator())
sub_axes.set_xlabel(
    r"Di-photon invariant mass $\mathrm{m_{\gamma\gamma}}$ [GeV]",
    x=1, horizontalalignment="right", fontsize=13,
)
sub_axes.tick_params(which="both", direction="in", top=True, right=True)
sub_axes.yaxis.set_minor_locator(AutoMinorLocator())

plt.show()

## Part 7: Clean up

Not strictly required for `cluster` - an idle pool is scaled to 0 and eventually deleted on its own (see `taskvine-gateway-test.ipynb` for details) - but this is immediate if you're done for sure.

In [ ]:
cluster.close()
del m
print("pool torn down, manager closed")